# ПЗ-5. Измерение галлюцинаций (POPE)

## 1. Цель
Построить сбалансированный тест наличия объектов и сравнить три промпта.

## 2. Датасеты и модели
100 изображений COCO val2017 с не менее чем тремя различными размеченными категориями. Частоты и совместная встречаемость вычисляются только по полным train2017 instance-аннотациям. Это POPE-подобный протокол, а не официальный leaderboard POPE.

[Qwen2.5-VL-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct), fp16.

Источник задания: `Практика_06_11_2026_1_week.docx`. Приложения А.1–А.7 отсутствуют; ниже приведены стандартные реализации. LoRA-проект не входит в эти шесть работ.

## 3. Подготовка окружения
Откройте этот notebook в Google Colab. Для ПЗ-1 выберите CPU, для ПЗ-3–6 — T4; для ПЗ-2 GPU ускорит получение признаков. Выполните установку, перезапустите сеанс, затем выполните остальные ячейки сверху вниз. Все результаты сохраняются в папке задания; скачайте её вместе с ноутбуком. Данные и веса кэшируются в соседней папке `data`. При переносе в Colab прежние результаты доступны только после загрузки файлов. GPU использует fp16. Числа в сохранённых выводах относятся только к реально выполненным ячейкам. Ячейки без execution_count не запускались.

In [ ]:
%pip install torch==2.9.1 torchvision==0.24.1 transformers==4.57.6 numpy==2.3.5 pandas==2.3.3 matplotlib==3.10.8 pillow==12.0.0 datasets==4.4.1 accelerate==1.12.0 sentencepiece==0.2.1 scikit-learn==1.7.2 pycocoevalcap==1.2 reportlab==4.4.5 huggingface-hub==0.36.2 safetensors==0.8.0 tokenizers==0.22.2 protobuf==6.33.2 ftfy==6.3.1 requests==2.34.2 fsspec==2025.10.0 pyarrow==26.0.1

In [1]:
LAB = 'PZ_5'
REVISIONS = {'facebook/dino-vitb16': 'f205d5d8e640a89a2b8ef0369670dfc37cc07fc2', 'openai/clip-vit-base-patch32': '3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268', 'google/siglip-base-patch16-224': '7fd15f0689c79d79e38b1c2e2e2370a7bf2761ed', 'Qwen/Qwen2.5-0.5B-Instruct': '7ae557604adf67be50417f59c2c2f167def9a775', 'Qwen/Qwen2.5-VL-3B-Instruct': '66285546d2b821cf421d4f5eb2576359d3770cd3', 'lmms-lab-encoder/DocVQA': '539088ef8a8ada01ac8e2e6d4e372586748a265e', 'lmms-lab-encoder/RefCOCO': 'a5dff0b3194715fda69b2d6a0d2aaafb41eaa407'}

import os
import sys
import json
import random
import time
import math
import gc
import hashlib
import platform
import importlib.metadata
from pathlib import Path
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor

ROOT = Path.cwd() if Path.cwd().name == LAB else Path.cwd() / LAB
ROOT.mkdir(exist_ok=True)
OUT = ROOT / "results"
OUT.mkdir(exist_ok=True)
DATA = ROOT.parent / "data"
DATA.mkdir(exist_ok=True)
os.environ["HF_HOME"] = str(DATA / "huggingface")
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(8, os.cpu_count() or 1))
torch.use_deterministic_algorithms(True, warn_only=True)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

def save_json(name, value):
    path = OUT / name
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    for attempt in range(20):
        try:
            temporary.replace(path)
            return
        except PermissionError:
            if attempt==19:
                raise
            time.sleep(0.1)

def save_figure(name):
    plt.savefig(OUT / name, dpi=130, bbox_inches="tight")
    plt.show()

versions = {p.metadata["Name"]: p.version for p in importlib.metadata.distributions() if p.metadata["Name"]}
environment = {"python": sys.version, "platform": platform.platform(), "seed": SEED,
               "device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
               "versions": versions, "revisions": REVISIONS}
save_json("environment.json", environment)
print({k: v for k, v in environment.items() if k != "versions"})

C:\Users\darri\PycharmProjects\cv061126\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26200-SP0', 'seed': 42, 'device': 'cpu', 'gpu': None, 'revisions': {'facebook/dino-vitb16': 'f205d5d8e640a89a2b8ef0369670dfc37cc07fc2', 'openai/clip-vit-base-patch32': '3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268', 'google/siglip-base-patch16-224': '7fd15f0689c79d79e38b1c2e2e2370a7bf2761ed', 'Qwen/Qwen2.5-0.5B-Instruct': '7ae557604adf67be50417f59c2c2f167def9a775', 'Qwen/Qwen2.5-VL-3B-Instruct': '66285546d2b821cf421d4f5eb2576359d3770cd3', 'lmms-lab-encoder/DocVQA': '539088ef8a8ada01ac8e2e6d4e372586748a265e', 'lmms-lab-encoder/RefCOCO': 'a5dff0b3194715fda69b2d6a0d2aaafb41eaa407'}}


In [2]:
from xml.sax.saxutils import escape
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image as PDFImage
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib import colors
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from matplotlib import font_manager

def write_report(title, goal, work, findings, table=None, figure=None):
    pdfmetrics.registerFont(TTFont("Student", font_manager.findfont("DejaVu Sans")))
    style = ParagraphStyle("Student", fontName="Student", fontSize=10, leading=14, spaceAfter=8)
    heading = ParagraphStyle("Heading", parent=style, fontSize=14, leading=18, spaceAfter=12)
    story = [Paragraph(escape(title), heading), Paragraph(escape("Цель. " + goal), style),
             Paragraph(escape("Что было сделано. " + work), style)]
    for text in findings:
        story.append(Paragraph(escape(text), style))
    if table is not None and len(table):
        rows = [[Paragraph(escape(str(x)), style) for x in table.columns]]
        rows += [[Paragraph(escape(f"{x:.4f}" if isinstance(x, float) else str(x)), style) for x in row]
                 for row in table.itertuples(index=False, name=None)]
        item = Table(rows, repeatRows=1, colWidths=[495 / len(table.columns)] * len(table.columns))
        item.setStyle(TableStyle([("VALIGN", (0,0), (-1,-1), "TOP"),
                                 ("GRID", (0,0), (-1,-1), 0.3, colors.grey)]))
        story.extend([item, Spacer(1, 10)])
    if figure is not None and Path(figure).exists():
        with Image.open(figure) as im:
            ratio = im.height / im.width
        story.append(PDFImage(str(figure), width=460, height=min(250, 460 * ratio)))
    story.append(Paragraph(escape("Индивидуальный вклад. Подготовлены реализации, протоколы экспериментов, тесты и оформление. Код и текст подготовлены с помощью ИИ-ассистента; фактические запуски и ограничения перечислены выше. Авторство невыполненных опытов не заявляется."), style))
    story.append(Paragraph(escape("Воспроизведение: notebook " + LAB + ".ipynb; seed=42. Версии среды и точные ревизии весов сохранены в results/environment.json. Контрольные вопросы разобраны в ноутбуке."), style))
    SimpleDocTemplate(str(ROOT / "report.pdf"), pagesize=(595,842), rightMargin=50, leftMargin=50,
                      topMargin=45, bottomMargin=45).build(story)
    print("PDF:", ROOT / "report.pdf")

In [3]:
import requests
from zipfile import ZipFile

def download(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        return destination
    temporary = destination.with_suffix(destination.suffix + ".part")
    for attempt in range(4):
        try:
            with requests.get(url, stream=True, timeout=(20, 120)) as response:
                response.raise_for_status()
                with temporary.open("wb") as stream:
                    for block in response.iter_content(1024 * 1024):
                        stream.write(block)
            temporary.replace(destination)
            return destination
        except requests.RequestException:
            if attempt == 3:
                raise
            time.sleep(attempt + 1)

def coco_annotations(year=2017):
    archive = download(f"https://s3.amazonaws.com/images.cocodataset.org/annotations/annotations_trainval{year}.zip",
                       DATA / f"annotations_trainval{year}.zip")
    with ZipFile(archive) as z:
        for name in z.namelist():
            if name.startswith("annotations/") and name.endswith(".json"):
                target = DATA / name
                if not target.exists():
                    target.parent.mkdir(exist_ok=True)
                    target.write_bytes(z.read(name))
    manifest = {"url": f"https://s3.amazonaws.com/images.cocodataset.org/annotations/annotations_trainval{year}.zip",
                "sha256": hashlib.file_digest(archive.open("rb"), "sha256").hexdigest()}
    save_json(f"coco{year}_source.json", manifest)


def coco_image(record, year=2017):
    split = record["split"]
    image_id = record["image_id"]
    name = f"{image_id:012d}.jpg" if year == 2017 else f"COCO_{split}{year}_{image_id:012d}.jpg"
    path = download(f"https://s3.amazonaws.com/images.cocodataset.org/{split}{year}/{name}", DATA / f"{split}{year}" / name)
    with Image.open(path) as image:
        return image.convert("RGB")


## 4. Реализация
В каждом режиме на изображение приходится 3 положительных и 3 отрицательных вопроса. Положительные объекты одинаковы во всех режимах. Popular ранжирует отсутствующие классы по частоте изображений train. Adversarial использует сумму P(кандидат | присутствующий класс), оценённых по train. Отсутствие подтверждается доступной разметкой, которая сама может быть неполной. Отбор изображений с ≥3 классами — ограничение протокола.

In [4]:
import random
from collections import Counter, defaultdict

def generate_questions(train_annotations, val_annotations, count=100, seed=42):
    categories = {c["id"]:c["name"] for c in train_annotations["categories"]}
    train_objects, val_objects = defaultdict(set), defaultdict(set)
    for ann in train_annotations["annotations"]:
        train_objects[ann["image_id"]].add(ann["category_id"])
    for ann in val_annotations["annotations"]:
        val_objects[ann["image_id"]].add(ann["category_id"])
    frequency, cooccurrence = Counter(), Counter()
    for present in train_objects.values():
        frequency.update(present)
        for first in present:
            for second in present:
                if first != second:
                    cooccurrence[first,second] += 1
    eligible = sorted(i for i, present in val_objects.items() if len(present)>=3 and len(categories)-len(present)>=3)
    rng = random.Random(seed)
    selected = rng.sample(eligible,count)
    rows = []
    for image_id in selected:
        present = val_objects[image_id]
        positive = rng.sample(sorted(present),3)
        absent = sorted(set(categories)-present)
        negatives = {
            "random":rng.sample(absent,3),
            "popular":sorted(absent,key=lambda c:(-frequency[c],c))[:3],
            "adversarial":sorted(absent,key=lambda c:(-sum(cooccurrence[p,c]/max(frequency[p],1) for p in present),c))[:3]
        }
        for mode, negative in negatives.items():
            assert len(set(negative))==3 and not (set(negative)&present)
            for label, objects in [(1,positive),(0,negative)]:
                for category_id in objects:
                    name = categories[category_id]
                    rows.append({"question_id":f"{image_id}_{mode}_{category_id}","image_id":image_id,
                                 "split":"val","mode":mode,"category_id":category_id,"object":name,
                                 "label":label,"question":f"Is there a {name} in the image?"})
    statistics = {"source":"COCO train2017 image-level object categories",
                  "train_images":len(train_objects),"seed":seed,"selected_images":selected,
                  "frequency":{str(k):v for k,v in frequency.items()},
                  "cooccurrence":{f"{a},{b}":v for (a,b),v in cooccurrence.items()}}
    return rows, statistics

In [5]:
import re

def normalize_yes_no(text):
    text = text.strip().lower()
    if re.search(r"\byes\s+(?:or|and)\s+no\b",text):
        return None
    match = re.match(r"^[\s*\"']*(yes|no|да|нет)\b",text)
    if not match:
        return None
    return int(match.group(1) in {"yes","да"})

def pope_metrics(rows):
    labels = np.array([r["label"] for r in rows])
    predictions = np.array([-1 if r["prediction"] is None else r["prediction"] for r in rows])
    tp = int(((labels==1)&(predictions==1)).sum())
    fp = int(((labels==0)&(predictions==1)).sum())
    tn = int(((labels==0)&(predictions==0)).sum())
    fn = int(((labels==1)&(predictions!=1)).sum())
    precision = tp/(tp+fp) if tp+fp else 0.
    recall = tp/(tp+fn) if tp+fn else 0.
    return {"Accuracy":(tp+tn)/len(rows),"Precision":precision,"Recall":recall,
            "F1":2*precision*recall/(precision+recall) if precision+recall else 0.,
            "yes_rate":float((predictions==1).mean()),"invalid_rate":float((predictions==-1).mean()),
            "false_positive_rate":fp/int((labels==0).sum()),"n":len(rows)}

In [6]:
assert normalize_yes_no("Yes, there is.")==1
assert normalize_yes_no("No.")==0
assert normalize_yes_no("yes or no") is None
assert normalize_yes_no("Maybe") is None
test_rows = [{"label":1,"prediction":1},{"label":1,"prediction":0},
             {"label":0,"prediction":1},{"label":0,"prediction":0}]
test_metrics = pope_metrics(test_rows)
assert all(test_metrics[k]==0.5 for k in ["Accuracy","Precision","Recall","F1","yes_rate"])
test_invalid = pope_metrics([{"label":1,"prediction":None},{"label":0,"prediction":None}])
assert test_invalid["Accuracy"]==0 and test_invalid["invalid_rate"]==1
save_json("unit_tests.json",{"normalization":True,"binary_metric_test":test_metrics,"invalid_penalized":True,"passed":True})
display(test_metrics)

{'Accuracy': 0.5,
 'Precision': 0.5,
 'Recall': 0.5,
 'F1': 0.5,
 'yes_rate': 0.5,
 'invalid_rate': 0.0,
 'false_positive_rate': 0.5,
 'n': 4}

## 5. Эксперименты
Генерация вопросов и статистика доступны на CPU. Для трёх режимов получается 1800 строк, а для трёх промптов — 5400 строк ответов. Все наборы сохраняются до инференса. Предварительное перечисление объектов получается от самой VLM, а не из аннотаций.

In [7]:
coco_annotations()
train_ann = json.loads((DATA/"annotations/instances_train2017.json").read_text())
val_ann = json.loads((DATA/"annotations/instances_val2017.json").read_text())
questions, object_statistics = generate_questions(train_ann,val_ann,100,SEED)
assert len(questions)==1800 and len({q["image_id"] for q in questions})==100
for mode in ["random","popular","adversarial"]:
    part = [r for r in questions if r["mode"]==mode]
    assert len(part)==600 and sum(r["label"] for r in part)==300
present_by_image = defaultdict(set)
for ann in val_ann["annotations"]:
    present_by_image[ann["image_id"]].add(ann["category_id"])
assert all((q["category_id"] in present_by_image[q["image_id"]])==bool(q["label"]) for q in questions)
save_json("questions.json",questions)
save_json("object_statistics.json",object_statistics)
balance = pd.DataFrame(questions).groupby(["mode","label"]).size().rename("questions").reset_index()
balance.to_csv(OUT/"balance.csv",index=False)
display(balance)

,mode,label,questions
0,adversarial,0,300
1,adversarial,1,300
2,popular,0,300
3,popular,1,300
4,random,0,300
5,random,1,300


In [8]:
def check_image(image_id):
    image = coco_image({"image_id":image_id,"split":"val"})
    return {"image_id":image_id,"width":image.width,"height":image.height,
            "rgb_sha256":hashlib.sha256(image.tobytes()).hexdigest()}
with ThreadPoolExecutor(max_workers=8) as pool:
    image_checks = list(pool.map(check_image,object_statistics["selected_images"]))
assert len(image_checks)==100
save_json("image_checks.json",image_checks)
print("Проверено оригинальных COCO val изображений:",len(image_checks))

Проверено оригинальных COCO val изображений: 100


In [9]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

def load_vlm(max_pixels=1000000):
    if DEVICE.type != "cuda":
        raise RuntimeError("Для полного эксперимента требуется GPU T4 16 GB или совместимая CUDA GPU. Размер выборки не уменьшен.")
    model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
    processor = AutoProcessor.from_pretrained(model_id, revision=REVISIONS[model_id],
                                             min_pixels=4*28*28, max_pixels=max_pixels, use_fast=False)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id, revision=REVISIONS[model_id], torch_dtype=torch.float16,
        attn_implementation="sdpa").to(DEVICE).eval()
    model.requires_grad_(False)
    return model, processor

def vlm_answer(model, processor, image, prompt, max_new_tokens=64):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    chat = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    batch = processor(text=[chat], images=[image], return_tensors="pt",
                      images_kwargs={"min_pixels":processor.image_processor.min_pixels,
                                     "max_pixels":processor.image_processor.max_pixels})
    grid = batch["image_grid_thw"][0].tolist()
    n_visual = int(np.prod(grid) // processor.image_processor.merge_size**2)
    batch = batch.to(DEVICE)
    batch["pixel_values"] = batch["pixel_values"].to(torch.float16)
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**batch, do_sample=False, max_new_tokens=max_new_tokens,
                                   use_cache=True)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    text = processor.batch_decode(generated[:, batch.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
    return {"answer": text, "visual_tokens": n_visual, "seconds": elapsed,
            "grid_thw": grid, "processed_height": grid[1]*14, "processed_width": grid[2]*14,
            "generated_tokens": generated.shape[1]-batch.input_ids.shape[1]}

In [ ]:
model, processor = load_vlm(max_pixels=1000000//784*784)
listing_file = OUT/"object_lists.json"
listings = json.loads(listing_file.read_text(encoding="utf-8")) if listing_file.exists() else {}
image_ids = object_statistics["selected_images"]
for image_id in tqdm(image_ids,desc="Object lists"):
    if str(image_id) not in listings:
        result = vlm_answer(model,processor,coco_image({"image_id":image_id,"split":"val"}),
                            "List the objects visible in this image. Be concise.",128)
        listings[str(image_id)] = result
        save_json("object_lists.json",listings)
answers_file = OUT/"answers.json"
answers = json.loads(answers_file.read_text(encoding="utf-8")) if answers_file.exists() else []
done = {(r["question_id"],r["prompt"]) for r in answers}
for prompt_name in ["base","uncertain_no","list_first"]:
    for image_id in tqdm(image_ids,desc=prompt_name):
        image = coco_image({"image_id":image_id,"split":"val"})
        for question in [r for r in questions if r["image_id"]==image_id]:
            if (question["question_id"],prompt_name) in done:
                continue
            text = question["question"]
            if prompt_name=="uncertain_no":
                text += " If you are not sure, answer no."
            elif prompt_name=="list_first":
                text = "You previously listed these objects: "+listings[str(image_id)]["answer"]+"\n"+text
            text += " Answer exactly Yes or No."
            result = vlm_answer(model,processor,image,text,24)
            parsed = normalize_yes_no(result["answer"])
            retry = None
            if parsed is None:
                retry = vlm_answer(model,processor,image,text+" Output only one word: yes or no.",8)
                parsed = normalize_yes_no(retry["answer"])
            answers.append({**question,"prompt":prompt_name,"prompt_text":text,**result,
                            "retry":retry,"prediction":parsed})
            save_json("answers.json",answers)

## 6. Полученные результаты
Нераспознанные ответы после одного повторного запроса считаются ошибками в Accuracy. Они не заменяются ответом no; отдельно показан invalid_rate. Recall учитывает пропуски положительных ответов, Precision — только фактические предсказания yes. Все сырые ответы, включая повторный запрос, сохранены.

In [ ]:
assert len(answers)==5400 and len({(r["question_id"],r["prompt"]) for r in answers})==5400
metric_rows = []
for mode in ["random","popular","adversarial"]:
    for prompt_name in ["base","uncertain_no","list_first"]:
        rows = [r for r in answers if r["mode"]==mode and r["prompt"]==prompt_name]
        assert len(rows)==600 and sum(r["label"] for r in rows)==300
        metric_rows.append({"mode":mode,"prompt":prompt_name,**pope_metrics(rows)})
metrics = pd.DataFrame(metric_rows)
metrics.to_csv(OUT/"metrics.csv",index=False)
display(metrics)
metrics.pivot(index="mode",columns="prompt",values="yes_rate").plot.bar(figsize=(8,4))
plt.axhline(0.5,color="black",linestyle="--")
plt.ylabel("Fraction of yes answers")
save_figure("yes_rate.png")

## 7. Анализ и выводы
Более высокий false positive rate для popular/adversarial по сравнению с random согласуется с влиянием языкового приора. Однако он не доказывает единственную причину ошибки: группы могут различаться визуальной сложностью, а COCO-разметка неполна. Инструкция answer no может уменьшить ложные yes ценой пропусков объектов; поэтому сравниваются и precision, и recall.

In [ ]:
notes = []
for prompt_name, group in metrics.groupby("prompt"):
    values = group.set_index("mode")
    delta = values.loc["adversarial","false_positive_rate"]-values.loc["random","false_positive_rate"]
    notes.append(f"Промпт {prompt_name}: FPR(adversarial)−FPR(random)={delta:.4f}.")
for mode,group in metrics.groupby("mode"):
    by_prompt = group.set_index("prompt")
    for prompt_name in ["uncertain_no","list_first"]:
        delta = by_prompt.loc[prompt_name,["Precision","Recall","F1"]]-by_prompt.loc["base",["Precision","Recall","F1"]]
        notes.append(f"{mode}, {prompt_name} относительно base: изменение Precision={delta.Precision:.4f}, Recall={delta.Recall:.4f}, F1={delta.F1:.4f}.")
notes.append("Связь с языковым приором интерпретируется как гипотеза, а не причинное доказательство.")
save_json("analysis.json",notes)
display(Markdown("\n\n".join(notes)))

## 8. Контрольные вопросы
**Почему adversarial труднее?** Отрицательные классы часто сопутствуют присутствующим в train, и модель может назвать ожидаемый по контексту, но отсутствующий объект.

**Чем полезна доля yes?** При балансе 1:1 она выявляет сдвиг ответов к согласию или отказу. Доля 0.5 сама по себе не означает высокую точность: так же может отвечать случайный классификатор.

In [10]:
findings = ["CPU-тесты нормализации yes/no и бинарных метрик пройдены. Синтетическая проверка даёт 0.5 для Accuracy, Precision, Recall, F1 и доли yes; это не результат VLM."]
if (OUT/"questions.json").exists():
    questions_saved = json.loads((OUT/"questions.json").read_text(encoding="utf-8"))
    statistics_saved = json.loads((OUT/"object_statistics.json").read_text(encoding="utf-8"))
    findings.append(f"По реальным COCO-аннотациям созданы {len(questions_saved)} вопросов для 100 изображений. Частоты и co-occurrence рассчитаны по {statistics_saved['train_images']} изображениям train. В каждом режиме 300 позитивных и 300 негативных вопросов; отсутствие негативов проверено по аннотациям.")
else:
    findings.append("Генератор вопросов реализован, но набор по COCO ещё не получен.")
if (OUT/"metrics.csv").exists():
    table = pd.read_csv(OUT/"metrics.csv")[["mode","prompt","Accuracy","Precision","Recall","F1","yes_rate"]]
    findings += json.loads((OUT/"analysis.json").read_text(encoding="utf-8"))
else:
    table = pd.read_csv(OUT/"balance.csv") if (OUT/"balance.csv").exists() else None
    findings += ["Ответы VLM для девяти сочетаний режима и промпта не получены: CUDA GPU недоступна. Итоговые Accuracy, Precision, Recall, F1 и доля yes неизвестны.",
                 "Код запускает предварительное перечисление объектов самой моделью, сохраняет сырые ответы, проверяет 5400 строк и строит таблицу 3×3. До запуска влияние промптов и языкового приора не установлено."]
write_report("ПЗ-5. POPE", "Измерить объектные галлюцинации и влияние промпта.",
             "Реализованы генератор random/popular/adversarial, нормализация ответов и полный протокол трёх промптов.",findings,table,OUT/"yes_rate.png")

PDF: C:\Users\darri\PycharmProjects\cv061126\PZ_5\report.pdf
